# 02 — Features usuario-día · CERT r4.2

**Objetivos de este notebook (Fase 2 del PLAN.md):**
1. Construir la tabla **usuario-día** (unidad de análisis de todo el proyecto) a partir de logon, device, file y email.
2. Añadir contexto estático: PC propio, rol/departamento (LDAP), rasgos psicométricos (Big Five).
3. Aplicar la **normalización doble UEBA**: z-score contra el histórico del propio usuario y contra su grupo de pares (rol).
4. Guardar el resultado en `data/processed/user_day_features.parquet`.

**Nota importante:** las etiquetas del ground truth (`label_malicious_day`, `is_insider_user`) se añaden **solo para evaluación posterior** de los modelos. Los modelos no supervisados (fase 3) no deben usarlas como input — son exclusivamente para medir precision/recall/F1 al final.

Ejecutable en **Google Colab** y en **local**, igual que `01_eda.ipynb`.

In [ ]:
# [compute]
# === Setup: entorno, dependencias y rutas ===
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    %pip install -q polars pyarrow
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/CERT_data")
    DATA_DIR = BASE / "r4.2"
    ANSWERS_DIR = BASE / "answers"
    PROCESSED_DIR = BASE / "processed"
else:
    # Local: el notebook vive en SIEM/notebooks/ — reutilizamos src/config.py
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    sys.path.insert(0, str(PROJECT_ROOT))
    from src.config import DATA_DIR, ANSWERS_DIR, PROCESSED_DIR

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

import polars as pl

DATE_FORMAT = "%m/%d/%Y %H:%M:%S"
WORK_HOURS = (8, 18)

print(f"Entorno: {'Colab' if IN_COLAB else 'Local'}")
print(f"Datos:   {DATA_DIR}")
print(f"Answers: {ANSWERS_DIR}")
print(f"Procesados: {PROCESSED_DIR}")

## Diseño de la tabla usuario-día

- **Unidad sparse**: solo se genera una fila `(user, day)` cuando el usuario tiene al menos un evento ese día en alguna de las fuentes (logon, device, file, email). No se rellenan los días sin actividad — evita una tabla densa de 1.000 usuarios × ~500 días (~500.000 filas con casi todo a cero) y simplifica el agregado vía outer-join.
- **"PC propio"** = el PC donde más veces hace `Logon` cada usuario (moda). Un `Logon` en otro PC (`n_other_pc_logons`) es una señal de comportamiento atípico.
- **Fuera de horario (`afterhours`)** = hora del evento `< 8` o `>= 18`, según `WORK_HOURS`. Se aplica igual a logon, device y file.
- **Normalización UEBA**: cada feature de conteo se transforma en dos z-scores, uno respecto a la media/desviación del propio usuario (`z_user_*`) y otro respecto a su rol (`z_role_*`). Esto es el núcleo del enfoque "Entity Behavior Analytics": lo anómalo se define en relación a la propia normalidad y a la de los pares.

In [ ]:
# [compute]
# === Funciones de carga ligera ===

def load_events(name: str, cols: list[str]) -> pl.LazyFrame:
    """Lazy-scan de un CSV de eventos, seleccionando solo las columnas necesarias
    y parseando la columna 'date' al formato datetime del proyecto."""
    return (
        pl.scan_csv(DATA_DIR / name)
        .select(cols)
        .with_columns(pl.col("date").str.strptime(pl.Datetime, DATE_FORMAT))
    )


def with_day_and_afterhours(lf: pl.LazyFrame) -> pl.LazyFrame:
    """Añade columnas derivadas de 'date': día (fecha), hora del día y flag
    de actividad fuera de horario laboral (WORK_HOURS)."""
    return lf.with_columns(
        pl.col("date").dt.date().alias("day"),
        pl.col("date").dt.hour().alias("hour"),
    ).with_columns(
        ((pl.col("hour") < WORK_HOURS[0]) | (pl.col("hour") >= WORK_HOURS[1])).alias("afterhours")
    )

In [ ]:
# [compute]
# === Features de logon ===

logon_lf = with_day_and_afterhours(load_events("logon.csv", ["date", "user", "pc", "activity"]))

# PC modal por usuario ("PC propio") = el pc con más Logon de cada usuario
pc_counts = (
    logon_lf.filter(pl.col("activity") == "Logon")
    .group_by("user", "pc")
    .len()
    .collect()
)
user_main_pc = (
    pc_counts.sort("len", descending=True)
    .group_by("user", maintain_order=True)
    .first()
    .select("user", pl.col("pc").alias("main_pc"))
)

logon_with_main = logon_lf.join(user_main_pc.lazy(), on="user", how="left")

logon_day = (
    logon_with_main.group_by("user", "day")
    .agg(
        (pl.col("activity") == "Logon").sum().alias("n_logons"),
        ((pl.col("activity") == "Logon") & pl.col("afterhours")).sum().alias("n_afterhours_logons"),
        pl.col("pc").filter(pl.col("activity") == "Logon").n_unique().alias("n_distinct_pcs"),
        ((pl.col("activity") == "Logon") & (pl.col("pc") != pl.col("main_pc"))).sum().alias("n_other_pc_logons"),
        pl.col("hour").filter(pl.col("activity") == "Logon").min().alias("first_logon_hour"),
        (pl.col("day").dt.weekday() >= 6).first().alias("is_weekend"),
    )
    .collect()
)

print(f"logon_day: {logon_day.shape}")
print(f"user_main_pc: {user_main_pc.shape}")
logon_day.head(3)

In [ ]:
# [compute]
# === Features de device (USB) y file ===

device_lf = with_day_and_afterhours(load_events("device.csv", ["date", "user", "pc", "activity"]))

device_day = (
    device_lf.group_by("user", "day")
    .agg(
        (pl.col("activity") == "Connect").sum().alias("n_usb_connects"),
        ((pl.col("activity") == "Connect") & pl.col("afterhours")).sum().alias("n_afterhours_usb"),
    )
    .collect()
)

# file.csv: solo date, user (sin content ni filename, para memoria)
file_lf = with_day_and_afterhours(load_events("file.csv", ["date", "user"]))

file_day = (
    file_lf.group_by("user", "day")
    .agg(
        pl.len().alias("n_file_copies"),
        pl.col("afterhours").sum().alias("n_afterhours_file_copies"),
    )
    .collect()
)

print(f"device_day: {device_day.shape}")
print(f"file_day:   {file_day.shape}")

In [ ]:
# [compute]
# === Features de email (streaming, sin 'content') ===

email_day = (
    pl.scan_csv(DATA_DIR / "email.csv")
    .select("date", "user", "to", "size", "attachments")
    .with_columns(
        pl.col("date").str.strptime(pl.Datetime, DATE_FORMAT),
        pl.col("to").fill_null(""),
    )
    .with_columns(
        pl.col("date").dt.date().alias("day"),
        pl.col("to").str.split(";").alias("to_list"),
    )
    .with_columns(
        (
            ~pl.col("to_list")
            .list.eval(pl.element().str.ends_with("@dtaa.com"))
            .list.all()
        ).alias("has_external"),
        pl.col("to_list").list.len().alias("n_recipients"),
    )
    .group_by("user", "day")
    .agg(
        pl.len().alias("n_emails"),
        pl.col("has_external").sum().alias("n_external_emails"),
        pl.col("attachments").sum().alias("total_attachments"),
        pl.col("size").sum().alias("total_email_size"),
        pl.col("n_recipients").sum().alias("n_recipients_total"),
    )
    .collect(engine="streaming")
)

print(f"email_day: {email_day.shape}")
email_day.head(3)

In [ ]:
# [compute]
# === Unión de fuentes + contexto estático (LDAP, psicometría) ===

features = (
    logon_day.join(device_day, on=["user", "day"], how="full", coalesce=True)
    .join(file_day, on=["user", "day"], how="full", coalesce=True)
    .join(email_day, on=["user", "day"], how="full", coalesce=True)
)

count_cols = [
    "n_logons", "n_afterhours_logons", "n_distinct_pcs", "n_other_pc_logons",
    "n_usb_connects", "n_afterhours_usb",
    "n_file_copies", "n_afterhours_file_copies",
    "n_emails", "n_external_emails", "total_attachments", "total_email_size", "n_recipients_total",
]
features = features.with_columns([pl.col(c).fill_null(0) for c in count_cols])
features = features.with_columns(
    pl.col("first_logon_hour").fill_null(-1),
    pl.col("is_weekend").fill_null(False),
)

# --- LDAP: primer snapshot + relleno desde snapshots posteriores ---
ldap_files = sorted((DATA_DIR / "LDAP").glob("*.csv"))
ldap_all = pl.concat([pl.read_csv(f) for f in ldap_files])
ldap_unique = ldap_all.unique(subset="user_id", keep="first")

ldap_ctx = (
    ldap_unique.select(
        pl.col("user_id").alias("user"),
        "role",
        "department",
    )
)

features = features.join(ldap_ctx, on="user", how="left")
features = features.with_columns(
    pl.col("role").fill_null("UNKNOWN"),
    pl.col("department").fill_null("UNKNOWN"),
)

# --- Psicometría (Big Five) ---
psychometric = pl.read_csv(DATA_DIR / "psychometric.csv").select(
    pl.col("user_id").alias("user"), "O", "C", "E", "A", "N"
)
features = features.join(psychometric, on="user", how="left")

print(f"features: {features.shape}")
features.head(3)

In [ ]:
# [compute]
# === Normalización UEBA: z-score propio (usuario) y de pares (rol) ===
#
# NOTA: estos z-scores se calculan con la media/desviación del PERIODO COMPLETO
# (ene-2010 a may-2011), incluyendo los días del propio incidente. Es aceptable
# para una detección no supervisada offline (no hay fuga de etiquetas, ya que
# no se usa el ground truth), pero infla ligeramente la varianza del usuario/rol
# durante los días anómalos, atenuando algo la señal. Se discute como limitación
# en la memoria; una alternativa sería una ventana rodante (rolling z-score).

COUNT_FEATURES = [
    "n_logons", "n_afterhours_logons", "n_distinct_pcs", "n_other_pc_logons",
    "n_usb_connects", "n_afterhours_usb",
    "n_file_copies", "n_afterhours_file_copies",
    "n_emails", "n_external_emails", "total_attachments", "total_email_size", "n_recipients_total",
]

z_exprs = []
for f in COUNT_FEATURES:
    z_user = (
        (pl.col(f) - pl.col(f).mean().over("user")) / pl.col(f).std().over("user")
    ).fill_null(0).fill_nan(0).alias(f"z_user_{f}")
    z_role = (
        (pl.col(f) - pl.col(f).mean().over("role")) / pl.col(f).std().over("role")
    ).fill_null(0).fill_nan(0).alias(f"z_role_{f}")
    z_exprs.extend([z_user, z_role])

features = features.with_columns(z_exprs)

print(f"features (con z-scores): {features.shape}")

In [ ]:
# [compute]
# === Etiquetas de evaluación (ground truth, solo para fase de evaluación) ===

insiders = (
    pl.read_csv(ANSWERS_DIR / "insiders.csv", schema_overrides={"dataset": pl.Utf8})
    .filter(pl.col("dataset") == "4.2")
    .with_columns(
        pl.col("start").str.strptime(pl.Datetime, DATE_FORMAT, strict=False),
        pl.col("end").str.strptime(pl.Datetime, DATE_FORMAT, strict=False),
    )
    .with_columns(
        pl.col("start").dt.date().alias("start_day"),
        pl.col("end").dt.date().alias("end_day"),
    )
    .select("user", "scenario", "start_day", "end_day")
)

INSIDER_USERS = set(insiders["user"].to_list())

features = features.join(insiders, on="user", how="left")
features = features.with_columns(
    (
        pl.col("start_day").is_not_null()
        & (pl.col("day") >= pl.col("start_day"))
        & (pl.col("day") <= pl.col("end_day"))
    ).fill_null(False).cast(pl.Int8).alias("label_malicious_day"),
    pl.col("scenario").fill_null(0),
    pl.col("user").is_in(list(INSIDER_USERS)).alias("is_insider_user"),
)

print(f"Total insiders r4.2: {insiders.height}")
print(f"features con etiquetas: {features.shape}")

In [ ]:
# [compute]
# === Guardar y validar ===

out_path = PROCESSED_DIR / "user_day_features.parquet"
features.write_parquet(out_path)

# --- Validaciones ---
n_users = features["user"].n_unique()
assert n_users == 1000, f"esperados 1000 usuarios, obtenidos {n_users}"

n_malicious_users = (
    features.filter(pl.col("label_malicious_day") == 1)["user"].n_unique()
)
assert n_malicious_users == 70, f"esperados 70 usuarios con día malicioso, obtenidos {n_malicious_users}"

assert 100_000 <= features.height <= 700_000, f"shape inesperado: {features.height} filas"

z_cols = [c for c in features.columns if c.startswith("z_user_") or c.startswith("z_role_")]
n_nulls_z = sum(features[c].null_count() for c in z_cols)
assert n_nulls_z == 0, f"hay {n_nulls_z} nulls en columnas z_*"

n_malicious_days = features["label_malicious_day"].sum()

print(f"Guardado en: {out_path}")
print(f"Shape final:                  {features.shape}")
print(f"Nº usuarios:                  {n_users}")
print(f"Usuarios con día malicioso:   {n_malicious_users}")
print(f"Total días maliciosos:        {n_malicious_days}")
print(f"Nulls en columnas z_*:        {n_nulls_z}")
print()
features.head(5)

## Conclusiones

El parquet `data/processed/user_day_features.parquet` contiene la tabla **usuario-día sparse**
(solo días con actividad registrada) con:

- **Features de comportamiento crudo**: logons, fuera de horario, PCs distintos, USB, copias de
  fichero a USB, emails (volumen, externos, adjuntos, tamaño, destinatarios).
- **Contexto estático**: PC propio implícito (vía `n_other_pc_logons`), rol/departamento (LDAP),
  rasgos psicométricos Big Five (O, C, E, A, N).
- **Normalización UEBA doble**: `z_user_*` (anomalía respecto al propio histórico) y `z_role_*`
  (anomalía respecto a los pares de rol), para cada feature de conteo.
- **Etiquetas de evaluación** (`label_malicious_day`, `scenario`, `is_insider_user`): derivadas del
  ground truth de `answers/insiders.csv`, **solo para medir precision/recall/F1** de los modelos
  no supervisados de la fase 3 — no deben usarse como input de los modelos.

**Limitación conocida**: los z-scores usan la media/desviación del periodo completo (incluyendo
los días de incidente), lo que puede atenuar ligeramente la señal anómala. Se documenta como
limitación y posible mejora futura (z-score rodante/ventana temporal).

**Siguiente fase (03_models)**: este parquet es el input directo de los modelos de detección de
anomalías (reglas baseline, Isolation Forest, autoencoder).